In [39]:
import json
from datetime import datetime, date, timedelta
from zoneinfo import ZoneInfo

import dspy

from connectors.google_calendar import GoogleCalendarConnector
#from connectors.clickup import ClickUpConnector
from config.settings import config

import requests
from typing import List, Dict, Optional

In [52]:
_calendar = GoogleCalendarConnector()
_clickup = ClickUpConnector()
_tz = ZoneInfo(config.TIMEZONE)

In [43]:
def get_today_calendar_events() -> str:
    """
    Obtiene todos los eventos del calendario de hoy.
    Retorna un JSON con la lista de reuniones (título, hora inicio, hora fin, asistentes).
    Úsala para saber qué reuniones tiene el usuario agendadas hoy.
    """
    events = _calendar.get_events_today()

    if not events:
        return json.dumps({"events": [], "message": "No hay eventos agendados para hoy."})

    parsed = []
    for e in events:
        parsed.append({
            "title": e["title"],
            "start": e["start"],
            "end": e["end"],
            "attendees": e["attendees"],
            "meet_link": e["meet_link"],
            "description": e["description"][:200] if e["description"] else "",
        })

    return json.dumps({"events": parsed, "total": len(parsed)}, ensure_ascii=False)


def create_clickup_task_for_meeting(title: str, start_time: str, end_time: str, attendees: str = "") -> str:
     """
    Crea una tarea en ClickUp para una reunión del calendario y registra
    automáticamente el tiempo de duración de la reunión en la tarea.
    Antes de crearla, verifica que no exista ya una tarea con ese nombre para evitar duplicados.

    Parámetros:
        title: Nombre de la reunión (será el nombre de la tarea)
        start_time: Hora de inicio en formato ISO 8601 (e.g. "2025-02-26T09:00:00-06:00")
        end_time: Hora de fin en formato ISO 8601
        attendees: Lista de asistentes separados por coma (opcional)

    Retorna un JSON indicando si la tarea fue creada, el tiempo registrado, o si ya existía.
    """
    # Calcular duración antes de cualquier cosa
    duration_min = _calculate_duration_minutes(start_time, end_time)
    duration_ms = duration_min * 60 * 1000

    # Verificar duplicado
    existing = _clickup.task_exists_with_name(title)
    if existing:
        return json.dumps({
            "status": "already_exists",
            "task_id": existing["id"],
            "message": f"Ya existe una tarea para '{title}', no se creó duplicado."
            })

    # Construir descripción
    description_parts = ["📅 Reunión agendada en Google Calendar"]
    if start_time:
        description_parts.append(f"🕐 Inicio: {start_time}")
    if end_time:
        description_parts.append(f"🕑 Fin: {end_time}")
    if duration_min:
        description_parts.append(f"⏱️ Duración: {duration_min} minutos")
    if attendees:
        description_parts.append(f"👥 Asistentes: {attendees}")

    description = "\n".join(description_parts)

    # due_date = hora de inicio de la reunión en ms
    due_date_ms = None
    try:
        dt = datetime.fromisoformat(start_time)
        due_date_ms = int(dt.timestamp() * 1000)
    except Exception:
        pass

    # Crear la tarea
    task = _clickup.create_task(
        name='DE: '+title,
        description=description,
        due_date=due_date_ms,
        tags=["reunión", "calendar-sync"],
    )
    task_id = task["id"]

    # Registrar tiempo de duración automáticamente
    time_logged = False
    if duration_ms > 0:
        try:
            _clickup.log_time(
                task_id=task_id,
                duration_ms=duration_ms,
                description=f"Duración de la reunión: {duration_min} min",
            )
            time_logged = True
        except Exception as e:
            print(f"⚠️  No se pudo registrar tiempo en tarea '{title}': {e}")

    return json.dumps({
        "status": "created",
        "task_id": task_id,
        "task_url": task.get("url", ""),
        "duration_minutes": duration_min,
        "time_logged": time_logged,
        "message": (
            f"Tarea '{title}' creada en ClickUp. "
            f"Duración registrada: {duration_min} min." if time_logged
            else f"Tarea '{title}' creada en ClickUp (sin tiempo registrado)."
        )
    })


def get_existing_clickup_tasks() -> str:
    """
    Obtiene las tareas actuales en ClickUp.
    Úsala para verificar qué tareas ya existen antes de crear nuevas,
    o para tener contexto de lo que ya está registrado.
    Retorna un JSON con la lista de tareas (id, nombre, status).
    """
    tasks = _clickup.get_all_tasks()
    simplified = [
        {
            "id": t["id"],
            "name": t["name"],
            "status": t.get("status", {}).get("status", "unknown"),
            "due_date": t.get("due_date", ""),
        }
        for t in tasks
    ]
    return json.dumps({"tasks": simplified, "total": len(simplified)}, ensure_ascii=False)

In [ ]:
SPRINT_ARKON_FIELD_ID = "d0c016df-e09a-492e-a7a2-cc92e1993627"

# Punto de anclaje: sprint c08 empezó el lunes 23 de febrero de 2026
# Cada sprint dura 2 semanas (14 días). A partir de aquí se calcula cualquier sprint.
SPRINT_ANCHOR_DATE = date(2026, 2, 23)  # lunes inicio de c08
SPRINT_ANCHOR_NUMBER = 8

SPRINT_OPTIONS: Dict[str, str] = {
    "2026c08": "cae799cd-e932-4f1e-91ba-a2b57c577b75",
    "2026c07": "277a358e-e245-47ae-9398-9283d50dca88",
    "2026c06": "23bdb71e-04dc-4622-85ce-5a47a375fd0d",
    "2026c05": "9a664fde-54c9-4fa5-afe8-77e6ec09e9b0",
    "2026c04": "bdedb19c-5d9c-4c6a-b09b-cdaf8b23c6b7"
}

def get_current_sprint_label(for_date: Optional[date] = None) -> str:
    """
    Calcula el label del sprint activo para una fecha dada (default: hoy).
    Lógica: desde el anclaje c08 (2026-02-23), cada sprint dura 14 días.
    Formato de salida: '2026c08', '2026c09', etc.
    """
    target = for_date or date.today()
    delta_days = (target - SPRINT_ANCHOR_DATE).days

    if delta_days < 0:
        # Fecha anterior al anclaje — calcular hacia atrás
        sprint_offset = -((-delta_days - 1) // 14 + 1)
    else:
        sprint_offset = delta_days // 14

    sprint_number = SPRINT_ANCHOR_NUMBER + sprint_offset
    year = target.year
    return f"{year}c{sprint_number:02d}"


def get_current_sprint_option_id(for_date: Optional[date] = None) -> Optional[str]:
    """
    Retorna el option_id del sprint activo.
    Retorna None si el sprint no está en SPRINT_OPTIONS todavía.
    """
    label = get_current_sprint_label(for_date)
    return SPRINT_OPTIONS.get(label)

In [51]:
class ClickUpConnector:
    BASE_URL = "https://api.clickup.com/api/v2"

    def __init__(self):
        self.headers = {
            "Authorization": config.CLICKUP_API_TOKEN,
            "Content-Type": "application/json",
            "accept": "application/json"
        }
        self.list_id = config.CLICKUP_LIST_ID

    def _get(self, endpoint: str, params: dict = None) -> dict:
        response = requests.get(
            f"{self.BASE_URL}/{endpoint}",
            headers=self.headers,
            params=params or {}
        )
        response.raise_for_status()
        return response.json()

    def _post(self, endpoint: str, payload: dict) -> dict:
        response = requests.post(
            f"{self.BASE_URL}/{endpoint}",
            headers=self.headers,
            json=payload
        )
        response.raise_for_status()
        return response.json()

    def _put(self, endpoint: str, payload: dict) -> dict:
        response = requests.put(
            f"{self.BASE_URL}/{endpoint}",
            headers=self.headers,
            json=payload
        )
        response.raise_for_status()
        return response.json()

     def create_task(
        self,
        name: str,
        description: str = "",
        due_date: Optional[int] = None,   # timestamp en ms
        tags: List[str] = None,
        set_current_sprint: bool = True,
    ) -> Dict:
        """Crea una tarea en la lista configurada.
        Asigna automáticamente el Sprint Arkon activo si set_current_sprint=True."""
        payload = {
            "name": name,
            "description": description,
            }
        if due_date:
            payload["due_date"] = due_date

        if set_current_sprint:
            sprint_label = get_current_sprint_label()
            sprint_id = get_current_sprint_option_id()
            if sprint_id:
                payload["custom_fields"] = [
                    {
                        "id": SPRINT_ARKON_FIELD_ID,
                        "value": [sprint_id],
                    }
                ]
                print(f"   📅 Sprint asignado: {sprint_label}")
            else:
                print(f"   ⚠️  Sprint '{sprint_label}' no tiene option_id — agrégalo a SPRINT_OPTIONS en clickup.py")

        task = self._post(f"list/{self.list_id}/task", payload)
        print(f"✅ Tarea creada en ClickUp: '{name}' (id: {task['id']})")
        return task

    def set_sprint(self, task_id: str, sprint_label: Optional[str] = None) -> bool:
            """
            Asigna Sprint Arkon a una tarea existente.
            Si sprint_label es None usa el sprint activo de hoy.
            """
            label = sprint_label or get_current_sprint_label()
            option_id = SPRINT_OPTIONS.get(label)

            if not option_id:
                print(f"⚠️  Sprint '{label}' no encontrado en SPRINT_OPTIONS")
                return False

            self._post(f"task/{task_id}/field/{SPRINT_ARKON_FIELD_ID}", {
                "value": [option_id]
            })
            print(f"✅ Sprint '{label}' asignado a tarea {task_id}")
            return True

    def log_time(self, task_id: str, duration_ms: int, description: str = "") -> Dict:
        """
        Registra tiempo en una tarea.
        duration_ms: duración en milisegundos (e.g. 3600000 = 1 hora)
        """
        from datetime import datetime
        payload = {
            "time": duration_ms,
            "description": description,
            "start": int(datetime.now().timestamp() * 1000),
            "end": int(datetime.now().timestamp() * 1000)+duration_ms
        }
        result = self._post(f"task/{task_id}/time", payload)
        print(f"✅ Tiempo registrado en tarea {task_id}: {duration_ms // 60000} min")
        return result

In [23]:
ev = get_today_calendar_events()

In [28]:
events = json.loads(ev)

In [25]:
tasks = get_existing_clickup_tasks()

In [27]:
tasks = json.loads(tasks)

In [29]:
events['events'][0]['title']

'Checkpoint Nexo 2'

In [30]:
events['events'][0]['start']

'2026-02-27T10:00:00-06:00'

In [31]:
events['events'][0]['end']

'2026-02-27T11:00:00-06:00'

In [32]:
events['events'][0]['attendees']

['jbarrios@grupoabraxas.com',
 'dalegria@grupoabraxas.com',
 'lvilchis@grupoabraxas.com',
 'rmeza@grupoabraxas.com',
 'fruiz@grupoabraxas.com',
 'chantal.aviles@grupobimbo.com',
 'diego.alegria@gbsupport.net',
 'francisco.ruiz@gbsupport.net',
 'jesus.barrios01@gbsupport.net',
 'juan.alvarez07@grupobimbo.com',
 'luis.vilchis@gbsupport.net',
 'ricardo.meza@gbsupport.net']

In [37]:
events

{'events': [{'title': 'Checkpoint Nexo 2',
   'start': '2026-02-27T10:00:00-06:00',
   'end': '2026-02-27T11:00:00-06:00',
   'attendees': ['jbarrios@grupoabraxas.com',
    'dalegria@grupoabraxas.com',
    'lvilchis@grupoabraxas.com',
    'rmeza@grupoabraxas.com',
    'fruiz@grupoabraxas.com',
    'chantal.aviles@grupobimbo.com',
    'diego.alegria@gbsupport.net',
    'francisco.ruiz@gbsupport.net',
    'jesus.barrios01@gbsupport.net',
    'juan.alvarez07@grupobimbo.com',
    'luis.vilchis@gbsupport.net',
    'ricardo.meza@gbsupport.net'],
   'meet_link': '',
   'description': '\n________________________________________________________________________________\nReunión de Microsoft Teams\nUnirse: https://teams.microsoft.com/meet/21607210510972?p=IOy63z4ZqH5TnkR2LQ\nId. de reunión'},
  {'title': 'Cortex | Daily Standup',
   'start': '2026-02-27T10:45:00-06:00',
   'end': '2026-02-27T11:00:00-06:00',
   'attendees': ['rmartinez@grupoabraxas.com',
    'hfigueroa@grupoabraxas.com',
    'fga

In [38]:
tasks

{'tasks': [{'id': '86b8nu54v',
   'name': 'PM / Reuniones semanales sprint 8 - 23 al 27 de febrero',
   'status': 'in progress',
   'due_date': '1772791200000'},
  {'id': '86b8mzr0q',
   'name': 'PM / Reuniones semanales sprint 8 - 23 al 27 de febrero',
   'status': 'in progress',
   'due_date': '1772186400000'},
  {'id': '86b8mczpm',
   'name': 'PM - Capybara - Reuniones Semanales',
   'status': 'in progress',
   'due_date': None},
  {'id': '86b8m9bgc',
   'name': 'PM /  Arkon Cortex | Review Sprint "8" - Fecha 06 Marzo',
   'status': 'Open',
   'due_date': '1772791200000'},
  {'id': '86b8m9bce',
   'name': 'PM/ Arkon Cortex | Dailys  Sprint 8 - 24 al 27 de febrero',
   'status': 'in progress',
   'due_date': '1772186400000'},
  {'id': '86b8ky4na',
   'name': 'Vac-Inc/ Vacaciones Cortex Marzo',
   'status': 'Open',
   'due_date': '1774951200000'},
  {'id': '86b8hvvhy',
   'name': 'PM /  Arkon Cortex | Planning  Sprint "8" - 23 Febrero',
   'status': 'in progress',
   'due_date': '1771